# 02 — Limpieza y consolidacion

**Objetivo:** Validar encuestas contra schemas y consolidar el dataset maestro.

**Librerias usadas:** jsonschema, pandas, pathlib

In [ ]:
import sys, json, os
from pathlib import Path
import pandas as pd
from jsonschema import validate, ValidationError

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'solineras'))
os.chdir(ROOT)

DATA = ROOT / 'data'
SCHEMAS = ROOT / 'solineras' / 'schemas'
pd.set_option('display.max_columns', 50)

## 1. Cargar schemas

In [ ]:
schemas = {}
for tipo, arch in [('solinera', 'schema_solinera.json'),
                   ('centro', 'schema_centro.json'),
                   ('residente', 'schema_residente.json')]:
    p = SCHEMAS / arch
    if p.exists():
        schemas[tipo] = json.load(open(p))
        print(f'OK schema {tipo}')
    else:
        print(f'FALTA schema {tipo}')

## 2. Leer y validar

In [ ]:
def leer_jsonl(p):
    if not p.exists():
        return []
    with open(p, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

def validar(doc, tipo):
    if tipo not in schemas:
        return True, None
    try:
        validate(instance=doc, schema=schemas[tipo])
        return True, None
    except ValidationError as e:
        return False, e.message

archivos = {
    'solinera': DATA / 'raw' / 'entrevistas_solineras.jsonl',
    'centro': DATA / 'raw' / 'entrevistas_centros.jsonl',
    'residente': DATA / 'raw' / 'entrevistas_residentes.jsonl',
}

validos, errores = {}, []
for tipo, path in archivos.items():
    lista = leer_jsonl(path)
    validos[tipo] = []
    for doc in lista:
        ok, msg = validar(doc, tipo)
        if ok:
            validos[tipo].append(doc)
        else:
            errores.append({'tipo': tipo, 'error': msg})

print('Validos:', {k: len(v) for k, v in validos.items()})
print('Errores:', len(errores))

## 3. Consolidar

In [ ]:
from consolidar import consolidar
df = consolidar()
if not df.empty:
    display(df.head())

## 4. Nulos

In [ ]:
if not df.empty:
    nulos = df.isna().sum().sort_values(ascending=False)
    print(nulos[nulos > 0])
else:
    print('Sin datos. Anade entrevistas en data/raw/')